In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ============================================================
# 1. PATHS
# ============================================================

ARTIFACTS_DIR = Path("artifacts")

FEATURE_DIR = ARTIFACTS_DIR / "05_feature_engineering"
OUTPUT_DIR = ARTIFACTS_DIR / "06_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_X_PATH = FEATURE_DIR / "05_train_features.npz"
VAL_X_PATH = FEATURE_DIR / "05_validation_features.npz"

TRAIN_Y_PATH = FEATURE_DIR / "05_train_labels.npy"
VAL_Y_PATH = FEATURE_DIR / "05_validation_labels.npy"

# IMPORTANT:
# Test files are intentionally NOT loaded here.
# They will only be loaded once, after model selection is finished.


# ============================================================
# 2. LOAD TRAIN + VALIDATION ONLY
# ============================================================

print("=" * 90)
print("LOADING TRAIN AND VALIDATION")
print("=" * 90)

X_train = sparse.load_npz(TRAIN_X_PATH)
X_val = sparse.load_npz(VAL_X_PATH)

y_train = np.load(TRAIN_Y_PATH)
y_val = np.load(VAL_Y_PATH)

print("Train features:", X_train.shape)
print("Validation features:", X_val.shape)

print("Train labels:", y_train.shape)
print("Validation labels:", y_val.shape)


# ============================================================
# 3. VALIDATION CHECKS
# ============================================================

assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)

assert X_train.shape[1] == X_val.shape[1]

assert set(np.unique(y_train)).issubset({0, 1})
assert set(np.unique(y_val)).issubset({0, 1})

print("\nInput validation: PASS")


# ============================================================
# 4. METRIC FUNCTION
# ============================================================

def classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_probability: np.ndarray,
) -> dict[str, float]:
    """
    Calculate classification metrics suitable for an imbalanced
    binary classification problem.
    """

    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                y_probability,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                y_probability,
            )
        ),
    }


# ============================================================
# 5. SIMPLE BASELINE
# ============================================================

print("\n" + "=" * 90)
print("BASELINE MODEL")
print("=" * 90)

baseline_model = DummyClassifier(
    strategy="most_frequent"
)

baseline_model.fit(
    X_train,
    y_train,
)

baseline_pred = baseline_model.predict(X_val)

baseline_probability = baseline_model.predict_proba(
    X_val
)[:, 1]

baseline_metrics = classification_metrics(
    y_true=y_val,
    y_pred=baseline_pred,
    y_probability=baseline_probability,
)

print("\nBaseline validation metrics:")

for metric, value in baseline_metrics.items():
    print(f"{metric:20s}: {value:.6f}")


# ============================================================
# 6. MODEL SEARCH
# ============================================================

print("\n" + "=" * 90)
print("LOGISTIC REGRESSION MODEL SEARCH")
print("=" * 90)

# We use the fixed validation set instead of random cross-validation
# because Notebook 3 created a chronological split.

C_VALUES = [
    0.01,
    0.1,
    1.0,
    10.0,
]

CLASS_WEIGHTS = [
    None,
    "balanced",
]

search_results: list[dict[str, object]] = []

best_model: LogisticRegression | None = None
best_validation_probability: np.ndarray | None = None
best_pr_auc = -np.inf
best_parameters: dict[str, object] | None = None

candidate_number = 0

for class_weight in CLASS_WEIGHTS:
    for c_value in C_VALUES:

        candidate_number += 1

        print(
            f"\nCandidate {candidate_number}: "
            f"C={c_value}, "
            f"class_weight={class_weight}"
        )

        model = LogisticRegression(
            C=c_value,
            class_weight=class_weight,
            solver="liblinear",
            max_iter=3000,
            random_state=42,
        )

        model.fit(
            X_train,
            y_train,
        )

        validation_probability = model.predict_proba(
            X_val
        )[:, 1]

        validation_prediction = (
            validation_probability >= 0.5
        ).astype("int8")

        metrics = classification_metrics(
            y_true=y_val,
            y_pred=validation_prediction,
            y_probability=validation_probability,
        )

        result = {
            "C": c_value,
            "class_weight": (
                "None"
                if class_weight is None
                else class_weight
            ),
            **metrics,
        }

        search_results.append(result)

        print(
            f"PR-AUC={metrics['pr_auc']:.6f} | "
            f"ROC-AUC={metrics['roc_auc']:.6f} | "
            f"F1={metrics['f1']:.6f} | "
            f"Recall={metrics['recall']:.6f}"
        )

        # Model selection metric:
        # PR-AUC because the positive class is rare.
        if metrics["pr_auc"] > best_pr_auc:

            best_pr_auc = metrics["pr_auc"]

            best_model = model

            best_validation_probability = (
                validation_probability
            )

            best_parameters = {
                "C": c_value,
                "class_weight": class_weight,
            }


# ============================================================
# 7. SAVE MODEL SEARCH RESULTS
# ============================================================

search_results_df = pd.DataFrame(
    search_results
)

search_results_df = search_results_df.sort_values(
    by="pr_auc",
    ascending=False,
).reset_index(drop=True)

search_results_df.to_csv(
    OUTPUT_DIR / "06_validation_model_search.csv",
    index=False,
)

print("\n" + "=" * 90)
print("MODEL SEARCH COMPLETE")
print("=" * 90)

print("\nBest parameters:")
print(best_parameters)

print(
    f"\nBest validation PR-AUC: "
    f"{best_pr_auc:.6f}"
)


# ============================================================
# 8. SAFETY CHECK
# ============================================================

if (
    best_model is None
    or best_validation_probability is None
    or best_parameters is None
):
    raise RuntimeError(
        "Model search did not produce a valid model."
    )


# ============================================================
# 9. THRESHOLD TUNING ON VALIDATION ONLY
# ============================================================

print("\n" + "=" * 90)
print("VALIDATION THRESHOLD TUNING")
print("=" * 90)

precision_values, recall_values, thresholds = (
    precision_recall_curve(
        y_val,
        best_validation_probability,
    )
)

# precision/recall contain one extra value compared to thresholds.
precision_for_thresholds = precision_values[:-1]
recall_for_thresholds = recall_values[:-1]

f1_values = (
    2
    * precision_for_thresholds
    * recall_for_thresholds
    / (
        precision_for_thresholds
        + recall_for_thresholds
        + 1e-12
    )
)

best_threshold_index = int(
    np.argmax(f1_values)
)

best_threshold = float(
    thresholds[best_threshold_index]
)

best_threshold_f1 = float(
    f1_values[best_threshold_index]
)

print(
    f"Best threshold: "
    f"{best_threshold:.6f}"
)

print(
    f"Validation F1 at selected threshold: "
    f"{best_threshold_f1:.6f}"
)


# ============================================================
# 10. VALIDATION METRICS WITH SELECTED THRESHOLD
# ============================================================

best_validation_prediction = (
    best_validation_probability
    >= best_threshold
).astype("int8")

best_validation_metrics = (
    classification_metrics(
        y_true=y_val,
        y_pred=best_validation_prediction,
        y_probability=best_validation_probability,
    )
)

print("\nBest validation metrics:")

for metric, value in best_validation_metrics.items():
    print(f"{metric:20s}: {value:.6f}")


# ============================================================
# 11. COMPARE AGAINST BASELINE
# ============================================================

print("\n" + "=" * 90)
print("BASELINE VS BEST MODEL")
print("=" * 90)

comparison_df = pd.DataFrame(
    [
        {
            "model": "Dummy Most Frequent",
            **baseline_metrics,
        },
        {
            "model": "Logistic Regression",
            **best_validation_metrics,
        },
    ]
)

print(
    comparison_df.to_string(
        index=False
    )
)

comparison_df.to_csv(
    OUTPUT_DIR
    / "06_baseline_vs_model_validation.csv",
    index=False,
)


# ============================================================
# 12. SAVE SELECTED MODEL BEFORE TEST
# ============================================================

joblib.dump(
    best_model,
    OUTPUT_DIR / "06_best_model.joblib",
)

print(
    "\nSelected model saved."
)


# ============================================================
# 13. TEST SET
# ============================================================
#
# IMPORTANT:
# Model selection and threshold tuning are now finished.
# Only now do we load the test set.
#
# The test set will be evaluated once.
# ============================================================

print("\n" + "=" * 90)
print("FINAL TEST EVALUATION")
print("=" * 90)

TEST_X_PATH = (
    FEATURE_DIR
    / "05_test_features.npz"
)

TEST_Y_PATH = (
    FEATURE_DIR
    / "05_test_labels.npy"
)

X_test = sparse.load_npz(
    TEST_X_PATH
)

y_test = np.load(
    TEST_Y_PATH
)

assert X_test.shape[0] == len(y_test)

assert (
    X_test.shape[1]
    == X_train.shape[1]
)

print(
    "Test features:",
    X_test.shape,
)

print(
    "Test labels:",
    y_test.shape,
)


# ============================================================
# 14. TEST PREDICTION
# ============================================================

test_probability = best_model.predict_proba(
    X_test
)[:, 1]

test_prediction = (
    test_probability
    >= best_threshold
).astype("int8")


# ============================================================
# 15. FINAL TEST METRICS
# ============================================================

test_metrics = classification_metrics(
    y_true=y_test,
    y_pred=test_prediction,
    y_probability=test_probability,
)

print("\nFinal TEST metrics:")

for metric, value in test_metrics.items():
    print(f"{metric:20s}: {value:.6f}")


# ============================================================
# 16. SAVE RESULTS SUMMARY
# ============================================================

results_summary = {
    "problem": (
        "Predict whether an order "
        "will be delivered late."
    ),
    "prediction_time": (
        "order_purchase_timestamp"
    ),
    "model": (
        "LogisticRegression"
    ),
    "model_selection_metric": (
        "PR-AUC"
    ),
    "threshold_selection_metric": (
        "F1 on validation"
    ),
    "best_parameters": {
        "C": float(
            best_parameters["C"]
        ),
        "class_weight": (
            "None"
            if best_parameters["class_weight"]
            is None
            else best_parameters[
                "class_weight"
            ]
        ),
    },
    "selected_threshold": (
        best_threshold
    ),
    "baseline_validation_metrics": (
        baseline_metrics
    ),
    "best_validation_metrics": (
        best_validation_metrics
    ),
    "test_metrics": (
        test_metrics
    ),
}

with open(
    OUTPUT_DIR
    / "06_results_summary.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        results_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 17. SAVE FINAL TEST PREDICTIONS
# ============================================================

test_predictions_df = pd.DataFrame(
    {
        "row_number": np.arange(
            len(y_test)
        ),
        "actual_is_late": y_test,
        "predicted_is_late": (
            test_prediction
        ),
        "late_probability": (
            test_probability
        ),
    }
)

test_predictions_df.to_csv(
    OUTPUT_DIR
    / "06_test_predictions.csv",
    index=False,
)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("NOTEBOOK 6 RESULT")
print("=" * 90)

print("SUCCESS")

print("\nBest model:")
print("  Logistic Regression")

print("\nBest parameters:")
print(
    f"  C = "
    f"{best_parameters['C']}"
)

print(
    f"  class_weight = "
    f"{best_parameters['class_weight']}"
)

print(
    f"\nSelected threshold: "
    f"{best_threshold:.6f}"
)

print(
    f"\nValidation PR-AUC: "
    f"{best_validation_metrics['pr_auc']:.6f}"
)

print(
    f"Test PR-AUC: "
    f"{test_metrics['pr_auc']:.6f}"
)

print(
    f"Test F1: "
    f"{test_metrics['f1']:.6f}"
)

print(
    f"Test Recall: "
    f"{test_metrics['recall']:.6f}"
)

print(
    f"Test Precision: "
    f"{test_metrics['precision']:.6f}"
)

print("\nArtifacts saved to:")
print(
    OUTPUT_DIR
)

for path in sorted(
    OUTPUT_DIR.iterdir()
):
    print(
        f"  - {path.name}"
    )

print(
    "\nNotebook 6 completed successfully."
)